# Markov-Chain Regime Model on BTC Daily Log-Returns

We discretize BTC daily log-returns into three regimes and fit a first-order Markov chain by maximum-likelihood (empirical transition counts).

**States**
- `DOWN` : log-return < -sigma
- `FLAT` : |log-return| <= sigma
- `UP`   : log-return > sigma

where `sigma` is the rolling 30-day stdev of log-returns (a state-aware threshold rather than a fixed cutoff).

**Outputs**
- 3x3 transition matrix `P`
- Stationary distribution `pi` such that `pi @ P = pi`
- Plot of the state sequence over time

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('../BTC_2019_2023_1d.csv')
df['datetime'] = pd.to_datetime(df['datetime'])
df.sort_values('datetime', inplace=True)
df['log_ret'] = np.log(df['close'] / df['close'].shift(1))
df['sigma']   = df['log_ret'].rolling(30).std()
df = df.dropna().reset_index(drop=True)

def classify(r, s):
    if r >  s: return 2   # UP
    if r < -s: return 0   # DOWN
    return 1              # FLAT

df['state'] = [classify(r, s) for r, s in zip(df['log_ret'], df['sigma'])]
df[['datetime', 'log_ret', 'sigma', 'state']].head()

In [ ]:
# Empirical transition counts -> row-stochastic transition matrix P.
K = 3
counts = np.zeros((K, K), dtype=float)
states = df['state'].to_numpy()
for a, b in zip(states[:-1], states[1:]):
    counts[a, b] += 1

P = counts / counts.sum(axis=1, keepdims=True)
labels = ['DOWN', 'FLAT', 'UP']
pd.DataFrame(P, index=labels, columns=labels).round(3)

In [ ]:
# Stationary distribution: left eigenvector of P with eigenvalue 1.
w, V = np.linalg.eig(P.T)
idx = np.argmin(np.abs(w - 1.0))
pi = np.real(V[:, idx])
pi = pi / pi.sum()
pd.Series(pi, index=labels, name='stationary').round(3)

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
ax[0].plot(df['datetime'], df['close'], color='black', linewidth=1)
ax[0].set_ylabel('BTC close')

colors = {0: 'red', 1: 'grey', 2: 'green'}
ax[1].scatter(df['datetime'], df['state'], c=[colors[s] for s in states], s=4)
ax[1].set_yticks([0, 1, 2]); ax[1].set_yticklabels(labels)
ax[1].set_xlabel('date'); ax[1].set_ylabel('regime')
plt.tight_layout(); plt.show()

## Interpretation

`P[i, j]` is the probability of moving from regime `i` to regime `j` on the next day. The diagonal entries quantify regime persistence; off-diagonals quantify the rate of mean-reversion across regimes. The stationary distribution `pi` is the long-run share of days spent in each regime under this first-order model.

A natural extension is a hidden-Markov variant where the states are latent and emission distributions are Gaussian over returns -- left out here to keep the model fully transparent.